# Work with Audio in ApertureDB

**Audio files are stored as [Blobs](https://docs.aperturedata.io/category/blob-commands) in ApertureDB. The query language lets you add, find, update, and delete audio blobs with rich metadata — dish name, contributor, cuisine — so you can retrieve exactly the clips you need.**


## Connect to ApertureDB

**Option A: ApertureDB Cloud (recommended)**  
Sign up for a [free 30-day trial](https://cloud.aperturedata.io). Get your key from **Connect > Generate API Key**, add it to a `.env` file in this directory:
```
APERTUREDB_KEY=your_key_here
```

**Option B: Community Edition (local Docker)**  
Run this in a terminal before starting the notebook:
```bash
docker run -d --name aperturedb \
  -p 55555:55555 -e ADB_MASTER_KEY=admin -e ADB_FORCE_SSL=false \
  aperturedata/aperturedb-community
```


In [ ]:
%pip install --upgrade --quiet aperturedb python-dotenv


In [ ]:
# Option A: ApertureDB Cloud
from dotenv import load_dotenv
load_dotenv()  # loads APERTUREDB_KEY from .env into the environment


In [ ]:
# Option B: Community Edition (local Docker)
# !adb config create localdb --active \
#     --host localhost --port 55555 \
#     --username admin --password admin \
#     --no-use-ssl --no-interactive


In [ ]:
from aperturedb.CommonLibrary import create_connector

client = create_connector()
response, _ = client.query([{"GetStatus": {}}])
client.print_last_response()


## Generate a Sample Audio Clip

For this example we generate a short WAV file — a cooking timer beep — using Python's built-in `wave` module. In practice, replace this with any `.wav`, `.mp3`, or other audio file from your recipe recordings.


In [ ]:
import wave, struct, math, os

os.makedirs("data", exist_ok=True)

sample_rate = 44100
duration    = 2       # seconds
frequency   = 880     # Hz — a short timer beep

# Apply a short fade-out to avoid clipping
samples = []
total = sample_rate * duration
for t in range(total):
    fade = max(0.0, 1.0 - t / total)  
    samples.append(int(32767 * fade * math.sin(2 * math.pi * frequency * t / sample_rate)))

with wave.open("data/cooking_timer.wav", "w") as f:
    f.setnchannels(1)
    f.setsampwidth(2)
    f.setframerate(sample_rate)
    f.writeframes(struct.pack("<" + "h" * len(samples), *samples))

print(f"Generated data/cooking_timer.wav ({os.path.getsize('data/cooking_timer.wav')} bytes)")


Preview the audio clip before storing it:


In [ ]:
from IPython.display import Audio
Audio("data/cooking_timer.wav")


## Add an Audio Blob to ApertureDB

Use [AddBlob](https://docs.aperturedata.io/query_language/Reference/blob_commands/AddBlob) to store the audio binary alongside metadata. Setting `type: audio` makes it easy to filter later.

**For bulk additions, use the [Python SDK loaders](https://docs.aperturedata.io/category/manage-multimodal-datasets).**


In [ ]:
query = [{
    "AddBlob": {
        "properties": {
            "document_type": "audio/wav",   # document_type lets the ApertureDB UI render the content appropriately
            "type":        "audio",
            "format":      "wav",
            "name":        "cooking_timer",
            "dish_name":   "Baked Potato",
            "cuisine":     "American",
            "contributor": "Gavin",
            "duration_s":  2,
        },
        "if_not_found": {"name": ["==", "cooking_timer"]},
    }
}]

with open("data/cooking_timer.wav", "rb") as f:
    audio_bytes = f.read()

response, _ = client.query(query, [audio_bytes])
client.print_last_response()


## Find Audio Blobs by Metadata

Use [FindBlob](https://docs.aperturedata.io/query_language/Reference/blob_commands/FindBlob) with constraints to retrieve audio clips. Set `blobs: true` to get the binary back.


In [ ]:
query = [{
    "FindBlob": {
        "constraints": {
            "type":    ["==", "audio"],
            "cuisine": ["==", "American"],
        },
        "blobs": True,
        "results": {"all_properties": True},
    }
}]

response, blobs = client.query(query)
client.print_last_response()


Play back the retrieved audio:


In [ ]:
from IPython.display import Audio
if blobs:
    display(Audio(blobs[0]))


## Update Audio Blob Metadata

Use [UpdateBlob](https://docs.aperturedata.io/query_language/Reference/blob_commands/UpdateBlob) to add or change properties on existing blobs.


In [ ]:
query = [{
    "UpdateBlob": {
        "constraints": {"name": ["==", "cooking_timer"]},
        "properties":  {"verified": True},
    }
}]

response, _ = client.query(query)
client.print_last_response()


## Delete the Audio Blob


In [ ]:
query = [{
    "DeleteBlob": {
        "constraints": {"name": ["==", "cooking_timer"]},
    }
}]

response, _ = client.query(query)
client.print_last_response()


## What's Next?

- [Work with Blobs](https://docs.aperturedata.io/HowToGuides/start/Blobs) — text files, PDFs, and other binary formats
- [Work with Videos](https://docs.aperturedata.io/HowToGuides/start/Videos) — add cooking video clips
- [Bulk load](https://docs.aperturedata.io/HowToGuides/Ingestion/Ingestion/Ingestion) — ingest many audio files at once with ParallelLoader
- [Add connections](https://docs.aperturedata.io/HowToGuides/start/Connections) — link audio clips to dish images or recipes
